# 02 — Join signal → risk → fills

Joins `signals`, `risk_decisions`, and `fills` on `correlation_id`.

**V2**: Every emitted signal must have a risk decision joinable by `correlation_id`.  
Note: `risk_decisions` and `fills` are empty until the engine submits live orders.
This notebook produces `joined.parquet` used by 03–07; it works with signal-only data.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src import db

In [ ]:
signals = pd.read_parquet('../data/raw/signals.parquet')
print(f'Signals: {len(signals)}')

In [ ]:
# Risk decisions (empty until live orders enabled)
risk_rows = db.query("""
    SELECT
        correlation_id,
        risk_level,
        rejection_reason  AS risk_rejection_reason,
        gate_latency_us,
        checked_at
    FROM risk_decisions
    WHERE correlation_id IS NOT NULL
""")
risk = pd.DataFrame(risk_rows) if risk_rows else pd.DataFrame(
    columns=['correlation_id','risk_level','risk_rejection_reason','gate_latency_us','checked_at']
)
print(f'Risk decisions: {len(risk)}')

In [ ]:
# Fills (empty until live orders enabled)
fill_rows = db.query("""
    SELECT
        correlation_id,
        fill_price,
        fill_size,
        remaining_size,
        submit_latency_us,
        fill_latency_us,
        venue_status,
        reject_reason     AS fill_reject_reason,
        submit_time,
        fill_time
    FROM fills
    WHERE correlation_id IS NOT NULL
""")
fills = pd.DataFrame(fill_rows) if fill_rows else pd.DataFrame(
    columns=['correlation_id','fill_price','fill_size','remaining_size',
             'submit_latency_us','fill_latency_us','venue_status',
             'fill_reject_reason','submit_time','fill_time']
)
print(f'Fills: {len(fills)}')

In [ ]:
joined = (
    signals
    .merge(risk,  on='correlation_id', how='left')
    .merge(fills, on='correlation_id', how='left')
)
print(f'Joined rows: {len(joined)}')
joined.head(3)

In [ ]:
# V2: every emitted signal should have a risk decision once live orders are enabled
emitted = joined[joined['decision'] == 'emitted']
missing_risk = emitted['risk_level'].isna()
print(f'Emitted signals : {len(emitted)}')
print(f'With risk record: {(~missing_risk).sum()}')
print(f'V2 gaps         : {missing_risk.sum()}  (expected if risk_decisions is empty)')

In [ ]:
joined.to_parquet('../data/raw/joined.parquet', index=False)
print('Saved data/raw/joined.parquet')